# Task 4.1 — Calibration draws

This notebook generates the calibration partitions (`task4/results/draws.json`).
 
**trial** is one five-target reach sequence from the go cue (`t_go`) to reward (`t_reward`)—not a single 50 ms bin. 

**ON trial**: recorded with amplitude-modulated (AM) stimulation active throughout (`stim_label: ON`, `stim_on_frac = 1`, session manifest `stim = 1`).

**OFF trial**: recorded with stimulation off. Session flags and trial labels agree across the dataset: all 48 stimulated sessions contain exclusively ON trials, and no session is mixed. **Included ON trials** (`include = 1`) are rewarded trials that passed QC (unrewarded trials are excluded).

For each stimulated test session and $k \in \{2, 5, 10, 20\}$, we sample subsets of $k$ included ON trials (**calibration draws**) from the 20-trial calibration pool (blocks 1–2)—10 distinct subsets for $k < 20$ and 1 full-pool draw for $k = 20$—and evaluate every draw on the same fixed 30-trial **query set** (blocks 3–5) defined in `splits.json`:

- **Independent draws across $k$:** Subsets are sampled independently for each $k$ without replacement (so draw $i$ at $k=2$ is not nested inside draw $i$ at $k=5$), isolating calibration set size $k$ as the sole variable.
- **Deterministic per-session seeding:** Each session is seeded independently, ensuring exact reproducibility and preventing added sessions from shifting existing partitions.
- **Leakage-free normalization:** Feature statistics are computed strictly from the $k$ drawn calibration trials, never from the held-out query set.


## Setup

In [ ]:
import os
import sys
import json
import pandas as pd

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
if TASK4_DIR not in sys.path:
    sys.path.insert(0, TASK4_DIR)

from utils import (
    TASK4_DIR,
    TASK4_RESULTS,
    DRAWS_PATH,
    RAW_VARIANT,
    LAG_COUNT,
    STIMULATION_ON,
    build_draws,
    save_draws,
    sessions_in_split,
    iter_draws,
    drawn_trial_windows,
    query_trial_windows,
)

print('task 4 helpers:', TASK4_DIR)
print('draws file   :', DRAWS_PATH)

task 4 helpers: /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4
draws file   : /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4/results/draws.json


## Sampling

Build the draws and print the invariant checks `build_draws` runs (set sizes, query sizes,
nestedness) before saving.

In [ ]:
SESSIONS = sessions_in_split('test', STIMULATION_ON) # get the test sessions with stimulation
DRAWS = build_draws(SESSIONS)
print(json.dumps(DRAWS['checks'], indent=1)[:600])
save_draws(DRAWS)
print('wrote', DRAWS_PATH)

{
 "passed": true,
 "failures": [],
 "notes": []
}
wrote /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4/results/draws.json


## Summary

One row per (session, k): how many distinct sets the 10 draws collapsed to, and how large
the query set is. Query shrinks as k grows — calibration trials come out of the same 50-trial
universe — and at k=20 only 30 query trials are left, which is what 4.3 scores.

In [ ]:
summary_rows = []
for session, entry in DRAWS['sessions'].items():  # one of SESSIONS
    for k in DRAWS['ks']:  # 2, 5, 10, 20 calibration trials
        draw_records = entry['subsets'][str(k)]  # the sampled draws at this k
        summary_rows.append(dict(session=session, condition=entry['condition'],
                                 am_freq_hz=entry['am_freq_hz'],
                                 n_on_trials=len(entry['on_trials']), k=k,
                                 # distinct trial sets, so a capped k is visible
                                 distinct_draws=len({frozenset(r['draw']) for r in draw_records}),
                                 query_trials=len(draw_records[0]['query']),
                                 n_draws=len(draw_records)))  # fixed query; k=20 has one draw
summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(TASK4_RESULTS, 'task4_1_draws_summary.csv'), index=False)
print('sessions:', summary.session.nunique(), '| rows:', len(summary_rows))
# the query is the same 30 trials at every k; k=20 has one full-pool draw
summary.groupby('k')[['query_trials', 'distinct_draws']].agg(['min', 'max'])

sessions: 14 | rows: 56


query_trials     distinct_draws    
            min max            min max
k                                     
2            48  48             10  10
5            45  45             10  10
10           40  40             10  10
20           30  30             10  10

## Check that one draw loads

Take the first draw of the first session and confirm the window loader turns it into
calibration and query arrays of the expected shape, with query + calibration = all ON trials.

In [ ]:
session = next(iter(DRAWS['sessions']))
calibration_size, draw_index, draw_record = next(iter_draws(DRAWS, session))
calibration_bin_features, calibration_bin_velocity = drawn_trial_windows(
    session, draw_record, RAW_VARIANT, LAG_COUNT)
query_bin_features, query_bin_velocity = query_trial_windows(
    session, draw_record, RAW_VARIANT, LAG_COUNT)
print(f'{session} k={calibration_size} draw={draw_index}')
print(f'  calibration bin rows {calibration_bin_features.shape} '
      f'-> velocity {calibration_bin_velocity.shape}')
print(f'  query bin rows       {query_bin_features.shape} '
      f'-> velocity {query_bin_velocity.shape}')
assert len(draw_record['draw']) == calibration_size
assert set(draw_record['draw']) <= set(DRAWS['sessions'][session]['pool'])
assert draw_record['query'] == DRAWS['sessions'][session]['query']

Butters-250520-101021 k=2 draw=0
  calibration windows (71, 2800) -> (71, 2)
  query windows      (1696, 2800) -> (1696, 2)
